# mT5 and XLM-R: which token count is correct?

**Background.** The adequacy notebook reproduced the paper's fertility exactly for eight tokenizers
but not for mT5 and XLM-R, which differ in 21 of the 41 files: Yoruba by about 19%, Malayalam by
9–11%, Persian by 4–7%. The fast and slow tokenizers in current Colab give identical counts, so the
switch that used to separate them no longer does anything (transformers 5 removed the separate slow
tokenizers).

**The test.** Both models were trained with Google's `sentencepiece` library, and their original
model files ship with them. Counting tokens with `sentencepiece` directly, bypassing transformers
entirely, gives the reference count. Whichever number it agrees with is the correct one.

| If sentencepiece matches | Then |
|---|---|
| the **published** numbers | the paper is right; current transformers under-counts. Pin `transformers<5` in the adequacy notebook and re-run Part A. Nothing in the paper changes. |
| the **current transformers** numbers | the August counts were too high, and the corrected per-file values replace them. |
| neither | something else differs. |

Section 5 is optional. It installs an older transformers in a separate environment to confirm what
produced the published numbers.

**Runtime:** CPU only, about 5 minutes (15 with Section 5). **Upload** `translations.zip` and
`results/superseded/fertility_production_v2_all11.csv` from this repository (`data/` and `results/`).

## 1. Setup and upload

In [ ]:
!pip -q install sentencepiece
import os, csv, glob, json, zipfile, subprocess, sys
import numpy as np, pandas as pd
import sentencepiece as spm, transformers
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
print("transformers", transformers.__version__, "| sentencepiece", spm.__version__)

try:
    from google.colab import files
    need = [f for f in ["translations.zip", "fertility_production_v2_all11.csv"] if not os.path.exists(f)]
    if need:
        print("Upload:", need); files.upload()
except ImportError:
    pass
for f in ["translations.zip", "fertility_production_v2_all11.csv"]:
    assert os.path.exists(f), f"missing {f}"
OUT = "results_spm"; os.makedirs(OUT, exist_ok=True)

## 2. Load the corpus
Same reader and naming as `01_tokenizer_fertility.ipynb`.

In [ ]:
if not os.path.exists("corpus"):
    with zipfile.ZipFile("translations.zip") as z:
        z.extractall("corpus")
csv_files = sorted(glob.glob("corpus/**/*.csv", recursive=True))

def load_csv(fp):
    keys = {}
    with open(fp, encoding="utf-8", errors="replace") as f:
        rdr = csv.reader(f); started = False; iS = iA = iT = None
        for row in rdr:
            if not started:
                low = [(c or "").strip().lower() for c in row]
                if "sura" in low and "aya" in low:
                    started = True; iS = low.index("sura"); iA = low.index("aya")
                    iT = low.index("translation") if "translation" in low else 3
                continue
            try:
                keys[(int(row[iS]), int(row[iA]))] = row[iT]
            except Exception:
                continue
    return keys

from collections import Counter
lang_name = lambda fp: os.path.basename(fp).split("_")[0]
cnt  = Counter(lang_name(f) for f in csv_files)
disp = {f: (lang_name(f) if cnt[lang_name(f)] == 1
            else lang_name(f) + "-" + os.path.basename(f).split("_")[1]) for f in csv_files}
TEXT = {disp[f]: load_csv(f) for f in csv_files}
NAMES = sorted(TEXT)
KEYS = sorted(set.intersection(*[set(v) for v in TEXT.values()]))
VERSES = {n: [str(TEXT[n][k]) for k in KEYS] for n in NAMES}
assert len(NAMES) == 41 and len(KEYS) == 6236, (len(NAMES), len(KEYS))
pub = pd.read_csv("fertility_production_v2_all11.csv").set_index("language").loc[NAMES]
print(len(NAMES), "files,", len(KEYS), "verses")

## 3. Count three ways

- `spm`: the original sentencepiece model file, encoded with the `sentencepiece` library
- `hf`: transformers as installed in this Colab, exactly as the paper's notebook called it
- `published`: the paper's value

A match means a maximum relative difference below 0.1% across all 41 files.

In [ ]:
MODELS = {"mt5":  ("google/mt5-small", "spiece.model"),
          "xlmr": ("xlm-roberta-base", "sentencepiece.bpe.model")}
rows, PERVERSE = [], {}
for col, (repo, fname) in MODELS.items():
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(repo, fname))
    hf = AutoTokenizer.from_pretrained(repo)
    for n in NAMES:
        v = VERSES[n]
        a = np.array([len(x) for x in sp.encode(v)], dtype=np.int32)
        b = np.array([len(x) for x in hf(v, add_special_tokens=False)["input_ids"]], dtype=np.int32)
        PERVERSE[f"{col}__spm__{n}"] = a
        rows.append({"tokenizer": col, "language": n, "published": pub.loc[n, f"{col}_tok_per_verse"],
                     "spm": a.mean(), "hf_now": b.mean(),
                     "bytes_per_verse": np.mean([len(x.encode("utf-8")) for x in v])})
    print(col, "done", flush=True)
R = pd.DataFrame(rows)
R["spm_vs_published"] = R.spm / R.published - 1
R["hf_vs_published"]  = R.hf_now / R.published - 1
R["spm_vs_hf"]        = R.spm / R.hf_now - 1
R.to_csv(f"{OUT}/counts_three_ways.csv", index=False)
np.savez_compressed(f"{OUT}/per_verse_spm.npz", **PERVERSE)

## 4. Verdict

In [ ]:
TOL = 1e-3
verdict = {}
for col in MODELS:
    d = R[R.tokenizer == col]
    m = {k: float(d[k].abs().max()) for k in ["spm_vs_published", "hf_vs_published", "spm_vs_hf"]}
    if m["spm_vs_published"] < TOL:
        v = "PUBLISHED IS CORRECT (sentencepiece matches the paper; current transformers is off)"
    elif m["spm_vs_hf"] < TOL:
        v = "PUBLISHED IS WRONG (sentencepiece matches current transformers; the paper's rows need correcting)"
    else:
        v = "NEITHER MATCHES (see the table below)"
    verdict[col] = {"verdict": v, **{k: round(x, 5) for k, x in m.items()}}
    print(f"\n=== {col}: {v}")
    print(f"    max |diff|  spm vs published {m['spm_vs_published']:.2e}   "
          f"current transformers vs published {m['hf_vs_published']:.2e}   spm vs current {m['spm_vs_hf']:.2e}")
    off = d[(d.spm_vs_published.abs() > TOL) | (d.spm_vs_hf.abs() > TOL)]
    if len(off):
        print(off[["language", "published", "spm", "hf_now", "spm_vs_published", "spm_vs_hf"]]
              .round(4).sort_values("spm_vs_published").to_string(index=False))

If sentencepiece disagrees with the paper, these are the numbers the paper's mT5 and XLM-R rows would
change to, computed the way `table_fertility.tex` computes them (41 translation files).

In [ ]:
p9010 = lambda x: np.percentile(x, 90) / np.percentile(x, 10)
rng = np.random.default_rng(42)
def boot_ci(x, stat, B=10000):
    x = np.asarray(x); s = [stat(x[rng.integers(0, len(x), len(x))]) for _ in range(B)]
    return np.percentile(s, [2.5, 97.5])
summ = []
for col in MODELS:
    d = R[R.tokenizer == col]
    for src in ["published", "spm", "hf_now"]:
        x = d[src].values
        lo, hi = boot_ci(x, np.mean); plo, phi = boot_ci(x, p9010)
        summ.append({"tokenizer": col, "source": src, "mean": round(x.mean(), 1),
                     "mean_ci": f"[{lo:.1f},{hi:.1f}]", "max_min": round(x.max() / x.min(), 1),
                     "p90p10": round(p9010(x), 3), "p90p10_ci": f"[{plo:.2f},{phi:.2f}]"})
SUMM = pd.DataFrame(summ)
SUMM.to_csv(f"{OUT}/table1_rows.csv", index=False)
print(SUMM.to_string(index=False))

## 5. Optional: confirm what produced the published numbers

Installs transformers 4.46 in a separate environment (your session is untouched) and counts the four
files that differ most. If it matches the published values, the paper's numbers came from the
transformers 4 conversion of these tokenizers.

In [ ]:
PROBE = ["yoruba", "malayalam", "persian", "assamese"]
json.dump({n: VERSES[n] for n in PROBE}, open("probe_verses.json", "w", encoding="utf-8"), ensure_ascii=False)
SCRIPT = r'''
import json, numpy as np, transformers
from transformers import AutoTokenizer
V = json.load(open("probe_verses.json", encoding="utf-8"))
out = {"version": transformers.__version__}
for col, repo in [("mt5", "google/mt5-small"), ("xlmr", "xlm-roberta-base")]:
    for fast in (True, False):
        tok = AutoTokenizer.from_pretrained(repo, use_fast=fast)
        out[f"{col}_fast{fast}"] = {n: float(np.mean([len(x) for x in tok(v, add_special_tokens=False)["input_ids"]]))
                                   for n, v in V.items()}
print(json.dumps(out))
'''
open("old_tf.py", "w").write(SCRIPT)
PY = "/content/tf4/bin/python"
if not os.path.exists(PY):
    subprocess.run(f"pip -q install uv && uv venv /content/tf4 -q && "
                   f"uv pip install -q --python {PY} 'transformers==4.46.3' sentencepiece protobuf 'numpy<2' torch "
                   f"--extra-index-url https://download.pytorch.org/whl/cpu", shell=True, check=True)
res = subprocess.run([PY, "-W", "ignore", "old_tf.py"], capture_output=True, text=True)
line = [l for l in res.stdout.splitlines() if l.startswith("{")]
if not line:
    print(res.stdout[-2000:], res.stderr[-2000:])
else:
    old = json.loads(line[-1]); print("transformers", old.pop("version"))
    rows = []
    for n in PROBE:
        for col in MODELS:
            rows.append({"language": n, "tokenizer": col, "published": pub.loc[n, f"{col}_tok_per_verse"],
                         "tf4_fast": old[f"{col}_fastTrue"][n], "tf4_slow": old[f"{col}_fastFalse"][n],
                         "spm": R[(R.tokenizer == col) & (R.language == n)].spm.iloc[0],
                         "tf_now": R[(R.tokenizer == col) & (R.language == n)].hf_now.iloc[0]})
    OLD = pd.DataFrame(rows).round(3); OLD.to_csv(f"{OUT}/transformers4_probe.csv", index=False)
    print(OLD.to_string(index=False))

## 6. Download
Outputs are collected in `results_spm.zip`.

In [ ]:
json.dump({"transformers_now": transformers.__version__, "verdict": verdict,
           "table1_rows": SUMM.to_dict(orient="records")},
          open(f"{OUT}/summary.json", "w"), indent=1)
print(json.dumps(verdict, indent=1))
import shutil; shutil.make_archive("results_spm", "zip", OUT)
try:
    from google.colab import files; files.download("results_spm.zip")
except Exception:
    print("written results_spm.zip")